# Sentiment Analysis

In [1]:
import os
print(os.path.abspath("arg_mining/ml_algorithms/ML/datasets/test.conll"))

c:\Users\huonc\Desktop\python\mining_project\arg_mining\ml_algorithms\ML\arg_mining\ml_algorithms\ML\datasets\test.conll


In [2]:
train_path = "datasets/train.conll"
test_path = "datasets/test.conll"
val_path = "datasets/validation.conll"
with open(train_path, encoding="utf-8") as f:
    train_data = f.read()

with open(test_path, encoding = "utf-8") as f:
    test_data = f.read()

with open(val_path, encoding = "utf-8") as f:
    val_data = f.read()

In [ ]:
#test with full dataset (no splitting)
full_data_path = "datasets/full_original_text.txt"
with open(full_data_path, encoding = "utf-8") as f:
    full_data = f.read()

'====================\nPost 1: CMV: I genuinely can\'t trust Israel on whatever they say anymore\nAuthor: ExtremeAcceptable289Post Text: So I\'ve been keeping up with Palestine news lately, and it\'s come to my attention that I feel I just can\'t trust Israel on anything anymore, even though it\'d be absurd to not trust them just because.\n\nThey\'ve lied on so many thing it\'s crazy:\n\nShereen Abu Akleh\n\nThe 40 beheaded babies (they also got Biden to lie about it)\n\nThe flour massacre\n\nThe al-shifa hospital incident in which an Israeli impersonated an al-Shifa doctor along with the edited video after Nov 2023 siege\n\nThe al-Ahli hospital faked voice call\n\nThe 15 executed aid workers \n\nHamas stealing aid (turns out an israeli funded gang did it)\n\nThe many, many times of "Palewood" lies (in which they later retacted/got debunked)\n\nThe gaza ministry of health being lies\n\nThe numbers of Hamas millitants dead (American intelligence and independent org says it is way less, 

In [ ]:
#clean txt full_data file
import re
def clean_text(data):
    #line separators
    data = re.sub(r'^\s*={5,}.*?\n', '', data, flags=re.MULTILINE)
    data = re.sub(r'^\s*\+{5,}.*?\n', '', data, flags=re.MULTILINE)

    #reddit structure
    data = re.sub(r'\[.*?\]\(.*?\)', '', data)
    data = re.sub(r'Post \d+:\s*CMV:', '', data)
    data = re.sub(r'Author:\s*[^P]+?Post Text:', '', data, flags=re.DOTALL)
    data = re.sub(r'Comment:', '', data)

    #reddit bot
    delta_patterns = [
        r'^.*DeltaBot.*$',
        r'^.*has awarded \d+ delta\(s\).*$',
        r'^.*All comments that earned deltas.*$',
        r'^.*Delta System Explained.*$',
        r'^.*Deltaboards.*$',
        r'^.*Please note that a change of view.*$',
    ]
    for pattern in delta_patterns:
        data = re.sub(pattern, '', data, flags=re.MULTILINE)
    data = re.sub(r'^\s*\d+\s*:\s*.*$', '', data, flags=re.MULTILINE)

    #specific punctuation
    data = re.sub(r'&gt;', ' ', data)
    data = re.sub(r'\^|\^', '', data) 
    data = re.sub(r'\|', '', data)   
    data = re.sub(r',,', '', data)   
    data = re.sub(r"''", '', data)   
    data = re.sub(r'[\'"]', '', data) #maybe affect the negation
    data = re.sub(r'[\u201E\u201C\-()]', '', data)
    data = re.sub(r'[,:;]', '', data)

    #keep whitespace
    data = re.sub(r'\s+', ' ',data).strip()

    return data

data = clean_text(full_data)

In [24]:
with open("datasets/full_text_cleaned.txt", encoding = "utf-8", mode = "w") as f:
    f.write(data)

In [25]:
import pandas as pd
def text_to_dataframe(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    df = pd.DataFrame(sentences, columns = ['sentences'])
    return df

df = text_to_dataframe(data)

In [26]:
df

,sentences
0,I genuinely cant trust Israel on whatever they...
1,Theyve lied on so many thing its crazy Shereen...
2,Your list conflates rumor Hamas propaganda and...
3,From the points it’s also unclear to me what y...
4,Furthermore who do you mean by Israel the gove...
...,...
69078,The thrill of discovery instantly ruined by th...
69079,Pass the mint.
69080,There will be wars over preserving these ancie...
69081,My shower thought addition to this is what if ...


In [ ]:
#with open("datasets/full_text_cleaned.txt", encoding = "utf-8", mode = "w") as f:
    #f.write(data)

In [3]:
#extract first column directly from conll files
def get_first_column(data):
    lines = data.strip().split('\n')
    first_col = []

    for i in lines:
        if i.strip():
            cols = i.split('\t')
            first_col.append(cols[0])

    return first_col

train_first_col = get_first_column(train_data)
test_first_col = get_first_column(test_data)
val_first_col = get_first_column(val_data)

len(train_first_col), len(test_first_col), len(val_first_col)


(943056, 236923, 1179979)

In [4]:
#join words to form sentence separated by tab
'''
open the conll file then remove whitespace, tabs, newline
there is only 1 column in the conll files so it'll just append those words into current_sentence
then join the current word to another word below to make a sentence
the final else indicates the last word of a sentence
'''
def read_conll_file(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.lower()
            line = line.strip()
            if line:
                parts = line.split('\t')
                word = parts[0]
                current_sentence.append(word)
            else:
                if current_sentence:
                    sentences.append(' '.join(current_sentence))
                    current_sentence = []

    if current_sentence:
        sentences.append(' '.join(current_sentence))
    return sentences

train_sentences = read_conll_file("datasets/train.conll")
test_sentences = read_conll_file("datasets/test.conll")
val_sentences = read_conll_file("datasets/validation.conll")
test_sentences[:10]

['comment i live in an agricultural area of my state i am appreciative of alllllll the crap alllll the farmers ranchers and producers deal with its alot',
 'generally american cities have a lot of really awesome neighborhoods the transit to get from one to the other is the commonly missing part and also lots of americans live in the suburbs which often wont have nice neighborhoods even',
 'comment americanflagscom some they are kind of pricy but were running a sale soon',
 '14 ronmckelvey',
 'there is something wrong with our culture and we need some serious introspection',
 'comment removed',
 'comment my sister had the knack for finding 4 leaf clovers she would find multiples all the time her husband worked a dangerous job and wore a 4 leaf clover for luck when the clover would wilt she would just go and find another no problem',
 '19 nkpstudios',
 'comment this is a subreddit for genuine discussion',
 '9 tiger0204']

In [5]:
#remove reddit usernames (elements start with a number)
train_sentences = [item for item in train_sentences if not item.split()[0].isdigit()]
test_sentences = [item for item in test_sentences if not item.split()[0].isdigit()]
val_sentences = [item for item in val_sentences if not item.split()[0].isdigit()]
train_sentences[:5]

['comment my grandparents greatestwhatever was before greatest generation used the gesture as well as regularly saying shame on you or you should be ashamed of yourself pretty often and it was very much considered the appropriate parenting stylei read a lot of early 20th century and late 19th century popular lit kids books dime novels magazine fiction and there were frequent scenes with similar language as an older gen x i heard it from grandparents and teachers and nuns but less from my parents silents as i think they had started to recognize that excessive guilt and shame is harmful',
 'comment its grey and cloudy today anyway',
 'comment i assume you meant greet',
 'violators will be fed to the bear',
 'i am not a personal fan of it on myself i literally dont care about it on anyone else though']

In [6]:
#vocab to keep
negation_words = [
    "not", "no", "never", "none", "nothing", "neither", "nor",
    "hardly", "scarcely", "barely", "without"
]
intensifiers = [
    "very", "really", "extremely", "quite", "so", "too", "just",
    "absolutely", "totally", "incredibly", "barely", "fairly", "almost", "nearly"
]
modal_verbs = [
    "could", "would", "should", "might", "may", "must", "can", "shall", "will"
]
auxiliary_verbs = [
    "is", "are", "was", "were", "be", "been", "being", "am",
    "do", "does", "did", "have", "has", "had"
]
pronouns = [
    "i", "you", "we", "they", "he", "she", "it",
    "me", "us", "them", "my", "your", "our", "their",
    "mine", "yours", "his", "hers", "its"
]
conjunctions = [
    "but", "although", "though", "yet", "while", "whereas"
]
subjective_adverbs = [
    "always", "never", "sometimes", "often", "seldom",
    "unfortunately", "fortunately", "luckily", "sadly", "happily"
]
exception_words = (
    negation_words +
    intensifiers +
    modal_verbs +
    auxiliary_verbs +
    pronouns +
    conjunctions +
    subjective_adverbs
)
exception_words[:10]

['not',
 'no',
 'never',
 'none',
 'nothing',
 'neither',
 'nor',
 'hardly',
 'scarcely',
 'barely']

### Preprocess Text

1. source text
* data cleaning
    * identify noise
    * noise removal
    * character normalization
    * data masking*
* linguistic processing
    * tokenization
    * POS tagging
    * stopwords
    * lemmatization
    * named-entity recognition
        

In [7]:
import pandas as pd
import nltk 
nltk.download('all')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    return " ".join(lemmatized_tokens)

train_processed = [preprocess_text(sentence) for sentence in train_sentences]
test_processed = [preprocess_text(sentence) for sentence in test_sentences]
val_processed = [preprocess_text(sentence) for sentence in val_sentences]

len(train_processed), len(test_processed), len(val_processed)

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\huonc\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\huonc\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\huonc\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\huonc\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\huonc\AppData\Roaming\nltk_data...
[

(29517, 7415, 36932)

In [8]:
#turn to pandas dataframe
df_train = pd.DataFrame({'sentences': train_processed})
df_test = pd.DataFrame({'sentences': test_processed})
df_val = pd.DataFrame({'sentences': val_processed})

In [9]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    if scores['pos'] > 0:
        sentiment = 1
    else:
        sentiment = 0
    return sentiment 

df_train['sentiment'] = df_train['sentences'].apply(get_sentiment)
df_test['sentiment'] = df_test['sentences'].apply(get_sentiment)
df_val['sentiment'] = df_val['sentences'].apply(get_sentiment)
    

In [10]:
from nltk.corpus import opinion_lexicon
negative_words = list(opinion_lexicon.negative())
negative_words[:10]

['2-faced',
 '2-faces',
 'abnormal',
 'abolish',
 'abominable',
 'abominably',
 'abominate',
 'abomination',
 'abort',
 'aborted']

In [11]:
#make a new column that shows positive sentence 
#expand contractions

%pip install contractions
%pip install swifter
import swifter
import contractions

negative_words = set(opinion_lexicon.negative())

def set_positive(sentence):
    expanded = contractions.fix(sentence)
    words = word_tokenize(expanded.lower())
    for i in words:
        if i in negative_words:
            return 0
    return 1

df_train['positive'] = df_train['sentences'].swifter.apply(set_positive)
df_test['positive'] = df_test['sentences'].swifter.apply(set_positive)
df_val['positive'] = df_val['sentences'].swifter.apply(set_positive)
    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Pandas Apply:   0%|          | 0/29517 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/7415 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/36932 [00:00<?, ?it/s]

In [12]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df_train['positive'], df_train['sentiment']))

[[ 3935 10234]
 [ 7909  7439]]


Confusion matrix: *many errors*

### Using VADER sentiment scoring

* bag of words

In [13]:
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm 

sia = SentimentIntensityAnalyzer()

In [14]:
sia.polarity_scores("bad")


{'neg': 1.0, 'neu': 0.0, 'pos': 0.0, 'compound': -0.5423}

In [15]:
df_train.reset_index(inplace = True)
df_test.reset_index(inplace = True)
df_val.reset_index(inplace = True)

In [16]:
#polarity score on the entire train dataset
result = {}
for i, row in tqdm(df_train.iterrows(), total = len(df_train)):
    text = row['sentences']
    myid = row['index']
    result[myid] = sia.polarity_scores(text)

  0%|          | 0/29517 [00:00<?, ?it/s]

In [17]:
vaders = pd.DataFrame(result).T
vaders = vaders.reset_index()
vaders = vaders.merge(df_train, how = "left")

In [18]:
vaders.head()

,index,neg,neu,pos,compound,sentences,sentiment,positive
0,0,0.158,0.639,0.203,0.4939,comment grandparent greatestwhatever greatest ...,1,0
1,1,0.000,0.769,0.231,0.0516,comment grey cloudy today anyway,1,0
2,2,0.000,0.566,0.434,0.3182,comment assume meant greet,1,1
3,3,0.630,0.370,0.000,-0.5267,violator fed bear,0,0
4,4,0.240,0.549,0.210,-0.0844,personal fan literally dont care anyone else t...,1,1


In [19]:
vaders.shape

(29517, 8)

### Roberta Pretrained Model

* train the dataset

In [20]:
#%pip install transformers
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

In [21]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

c:\Users\huonc\Desktop\python\mining_project\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\huonc\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [22]:
#vader
test = df_train['sentences'][50]
print(test)
sia.polarity_scores(test)

comment youd assume position get fucked small peasant revolt actual witch hunter hearing murmuring know sign look determining one witch legitimate wizard one messing thing


{'neg': 0.299, 'neu': 0.701, 'pos': 0.0, 'compound': -0.8555}

In [29]:
def polarity_scores_roberta(example):
    #roberta model
    encoded_text = tokenizer(test, return_tensors="pt")
    out = model(**encoded_text)
    scores = out[0][0].detach().numpy()
    scores = softmax(scores) #neg, neu, pos
    scores_dict = {
        'roberta_neg': scores[0],
        'roberta_neu': scores[1],
        'roberta_pos': scores[2]
    }
    scores_dict

    return scores_dict



In [33]:
#polarity score on the entire train dataset
result = {}
for i, row in tqdm(df_train.iterrows(), total = len(df_train)):
    text = row['sentences']
    myid = row['index']
    vader_result = sia.polarity_scores(text)

    vader_result_rename = {}
    for key, value in vader_result.items():
        vader_result_rename[f"vader_{key}"] = value
    roberta_result = polarity_scores_roberta(text)

    both = {**vader_result, **roberta_result}
    result[myid] = both
    

  0%|          | 0/29517 [00:00<?, ?it/s]

In [ ]:
both

{'neg': 0.158,
 'neu': 0.639,
 'pos': 0.203,
 'compound': 0.4939,
 'roberta_neg': 0.542965,
 'roberta_neu': 0.44426653,
 'roberta_pos': 0.012768475}

In [34]:
results_df = pd.DataFrame(result).T
results_df = results_df.reset_index()
results_df = results_df.merge(df_train, how = "left")

In [35]:
results_df.head()

,index,neg,neu,pos,compound,roberta_neg,roberta_neu,roberta_pos,sentences,sentiment,positive
0,0,0.158,0.639,0.203,0.4939,0.542965,0.444267,0.012768,comment grandparent greatestwhatever greatest ...,1,0
1,1,0.000,0.769,0.231,0.0516,0.542965,0.444267,0.012768,comment grey cloudy today anyway,1,0
2,2,0.000,0.566,0.434,0.3182,0.542965,0.444267,0.012768,comment assume meant greet,1,1
3,3,0.630,0.370,0.000,-0.5267,0.542965,0.444267,0.012768,violator fed bear,0,0
4,4,0.240,0.549,0.210,-0.0844,0.542965,0.444267,0.012768,personal fan literally dont care anyone else t...,1,1


In [36]:
from transformers import pipeline 

sentiment_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\huonc\Desktop\python\mining_project\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\huonc\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' packa

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [37]:
sentiment_pipeline("not bad, it's just ok")

[{'label': 'POSITIVE', 'score': 0.9998577833175659}]

In [40]:
sentiment_pipeline("not bad, he's a successfully failed the mission")

[{'label': 'POSITIVE', 'score': 0.9959104061126709}]